In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG = timeSeriesData_BIGraw.copy()
userInputData = userInputDataRaw.copy()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space and the door is closed
room_mask = userInputData["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0', 'Κρεβατοκάμαρα Όλοι οι αισθητήρες μαζί Π:1.80 , Δ:2.00'])

open_door_mask = userInputData["are-doors-opened"] != "on"
mask = room_mask & open_door_mask
userInputData = userInputData.loc[mask]
#grab all the data that are contained in those experiments
timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}
for index,data in dict_of_timeseries.items():
    dict_of_timeseries[index] = dict_of_timeseries[index].set_index("seconds",drop=False)

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 

        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        

        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
           
                #value to show the time that source is inserted
          
            userInputDataRow = userInputData.loc[key_value,:]
        #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
        #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
            
            euclidian_distances = (
                                  f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
            )
            subtitle=  (
                        f"At experiment with key {key_value}\n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n", 
                        f"experimentState:{userInputDataRow['experimentState']}\n",
                        f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
            )
            if ("distance"  in type_of_other_grouping):
               subtitle = subtitle + euclidian_distances  
            ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
                    top=0.75,   # space for overall title
                    wspace=0.2, # horizontal space between subplots
                    hspace=0.3 # vertical space between subplots
                   )

        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)        

plot_all_positions(userInputData)

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

sensors = timeSeriesData_BIG["sensors"].unique()
euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
group_by_list = ["room","experimentState",*euclidian_distances_columns]
room_other_grouping = userInputData.groupby(group_by_list).groups
type_of_other_grouping = ["experimentState","position","distance"]

sensors = timeSeriesData_BIG["sensors"].unique()
for sensor in sensors:
    mask = timeSeriesData_BIG["sensors"] == sensor
    timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
    print(sensor)
    printDataBasedOnDate("VOC rolling average",userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

In [ ]:
room_mask = userInputData["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
experiment_state_mask = userInputData["experimentState"] == "InsertingSourcePollutant"
userInputDataModel = userInputData.loc[room_mask & experiment_state_mask].copy()
df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputDataModel.index)]
dfs_by_sensor = {
    sensor: g
    for sensor, g in df_filtered.groupby("sensors")
}


In [ ]:
dfs_by_sensor

In [ ]:
dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].columns

In [ ]:
userInputDataModel

In [ ]:
sensors = timeSeriesData_BIG["sensors"].unique()

euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
userInputDataModel.loc[:,euclidian_distances_columns] = userInputDataModel.loc[:,euclidian_distances_columns].round(2)

In [ ]:
userInputDataModel[euclidian_distances_columns].describe().loc[["min","max"]]

In [ ]:
for column in euclidian_distances_columns:
    print(userInputDataModel[column].unique())

In [ ]:
columns_to_keep = ["VOC rolling average"]

key_to_grab_size = userInputDataModel.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_keep]
print(sample_data.shape[0])


In [ ]:


columns_to_keep = ["VOC"]

key_to_grab_size = userInputDataModel.index[0]
mask = dfs_by_sensor['Id=0:BME680:breathVocEquivalent']["keys"] == key_to_grab_size

sample_data = dfs_by_sensor['Id=0:BME680:breathVocEquivalent'].loc[mask,columns_to_keep]

X_subset_columns_size = sample_data.shape[0]


sensors = timeSeriesData_BIG["sensors"].unique()
X ={}
Y ={}


dict_flattened_arrays_per_sensor_per_distance ={}
for sensor in sensors:
    X[sensor] = np.empty((0, X_subset_columns_size))
    Y[sensor] = np.empty((0, 1))
    dict_flattened_arrays_per_sensor_per_distance[sensor] = {}
    euclidian_distance_column = f"Euclidian distance to {sensor}"    
    
    for distance,experiments in userInputDataModel.groupby(euclidian_distance_column):

        rows_size = experiments.shape[0]
        X_distance_subset = np.empty((rows_size,X_subset_columns_size))

        for array_index,experiment_index in enumerate(experiments.index):
          
            mask = dfs_by_sensor[sensor]["keys"]== experiment_index
            flatten_array = dfs_by_sensor[sensor].loc[mask,columns_to_keep].to_numpy().reshape(1, -1)
            
            X_distance_subset[array_index,:] =   flatten_array
        X[sensor] = np.vstack((X[sensor],X_distance_subset))   
            

        Y_distance_subset = np.full((rows_size,1),distance)
        
        Y[sensor] = np.vstack((Y[sensor],Y_distance_subset))   


In [ ]:
X

In [ ]:
Y

In [ ]:
X.keys()

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Step 1: Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X['Id=0:BME680:breathVocEquivalent'])

# Step 2: Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Step 3: Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Only display first 10 components max
max_components = min(10, len(explained_variance))
ev_to_plot = explained_variance[:max_components]
cum_to_plot = cumulative_variance[:max_components]

# Step 4: Bar plot of cumulative explained variance
plt.figure(figsize=(8,5))
plt.bar(range(1, max_components + 1), cum_to_plot)
plt.xlabel('Number of PCA components')
plt.ylabel('Cumulative explained variance')
plt.title('Cumulative explained variance (first 10 components)')
plt.grid(True, axis='y')
plt.show()

# Step 5: Optimal number of components for ~90% variance
optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
print("Optimal number of components to explain ~90% variance:", optimal_components)


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Step 1: Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X['Id=0:BME680:breathVocEquivalent'])

# Step 2: Apply PCA with 5 components
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

# Explained variance for each of the 5 components
explained_variance = pca.explained_variance_ratio_

# Step 3: Bar plot of explained variance (per component)
plt.figure(figsize=(8,5))
plt.bar(range(1, 6), explained_variance)
plt.xlabel('PCA Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by PCA Components (5 components)')
plt.grid(True, axis='y')
plt.show()

'Id=0:BME680:breathVocEquivalent'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# --- Config ---
n_components = 5         # number of PCA components you chose
test_size = 0.2           # fraction for test set
random_state = 42         # predefined random state for reproducibility

# --- Get arrays (ensure they're numpy arrays) ---
X_arr = np.asarray(X['Id=0:BME680:breathVocEquivalent'])
y_arr = np.asarray(Y['Id=0:BME680:breathVocEquivalent'])


# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=test_size, random_state=random_state, shuffle=True
)

# --- Scale: fit on train, transform train & test ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler.transform(X_test)        # transform only

# --- PCA: fit on scaled train, transform both ---
#pca = PCA(n_components=n_components, random_state=random_state)
#X_train_pca = pca.fit_transform(X_train_scaled)
#X_test_pca  = pca.transform(X_test_scaled)

# --- SVR: train on PCA data ---
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)  # tune C, epsilon, kernel as needed
svr.fit(X_train_scaled, y_train)

# --- Predict & evaluate ---
y_pred = svr.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE on test set: {mse:.4f}")
print(f"R^2 on test set: {r2:.4f}")

# --- Plot: predicted vs actual ---
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title("SVR predictions vs actual (after scaling + PCA)")
plt.grid(True)
plt.show()


In [ ]:
y_test

In [ ]:
y_pred

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_params = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}

svr = GridSearchCV(
    estimator=SVR(kernel='rbf'),
    param_grid=svr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

svr.fit(X_train_pca, y_train)
print("SVR best params:", svr.best_params_)

from sklearn.metrics import mean_squared_error, r2_score

best_model = svr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________



from sklearn.ensemble import RandomForestRegressor

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10]
}

rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

rf.fit(X_train_pca, y_train)
print("Random Forest best params:", rf.best_params_)


best_model = rf.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.ensemble import GradientBoostingRegressor

gbr_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

gbr = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gbr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

gbr.fit(X_train_pca, y_train)
print("GradientBoosting best params:", gbr.best_params_)

best_model = gbr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.neighbors import KNeighborsRegressor

knn_params = {
    "n_neighbors": [2, 3, 4, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]   # Manhattan / Euclidean
}

knn = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=knn_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

knn.fit(X_train_pca, y_train)
print("KNN best params:", knn.best_params_)


best_model = knn.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Ridge

ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

ridge.fit(X_train_pca, y_train)
print("Ridge best params:", ridge.best_params_)

best_model = ridge.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Lasso

lasso_params = {
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1]
}

lasso = GridSearchCV(
    estimator=Lasso(max_iter=10000),
    param_grid=lasso_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

lasso.fit(X_train_pca, y_train)
print("Lasso best params:", lasso.best_params_)


best_model = lasso.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


'Id=1:BME680:breathVocEquivalent'

In [ ]:
# --- Config ---
n_components = 10        # number of PCA components you chose
test_size = 0.2           # fraction for test set
random_state = 42         # predefined random state for reproducibility

# --- Get arrays (ensure they're numpy arrays) ---
X_arr = np.asarray(X['Id=1:BME680:breathVocEquivalent'])
y_arr = np.asarray(Y['Id=1:BME680:breathVocEquivalent'])


# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=test_size, random_state=random_state, shuffle=True
)

# --- Scale: fit on train, transform train & test ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler.transform(X_test)        # transform only

# --- PCA: fit on scaled train, transform both ---
pca = PCA(n_components=n_components, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# --- SVR: train on PCA data ---
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)  # tune C, epsilon, kernel as needed
svr.fit(X_train_pca, y_train)

# --- Predict & evaluate ---
y_pred = svr.predict(X_test_pca)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE on test set: {mse:.4f}")
print(f"R^2 on test set: {r2:.4f}")

# --- Plot: predicted vs actual ---
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title("SVR predictions vs actual (after scaling + PCA)")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_params = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}

svr = GridSearchCV(
    estimator=SVR(kernel='rbf'),
    param_grid=svr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

svr.fit(X_train_pca, y_train)
print("SVR best params:", svr.best_params_)

from sklearn.metrics import mean_squared_error, r2_score

best_model = svr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________



from sklearn.ensemble import RandomForestRegressor

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10]
}

rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

rf.fit(X_train_pca, y_train)
print("Random Forest best params:", rf.best_params_)


best_model = rf.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.ensemble import GradientBoostingRegressor

gbr_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

gbr = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gbr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

gbr.fit(X_train_pca, y_train)
print("GradientBoosting best params:", gbr.best_params_)

best_model = gbr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.neighbors import KNeighborsRegressor

knn_params = {
    "n_neighbors": [2, 3, 4, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]   # Manhattan / Euclidean
}

knn = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=knn_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

knn.fit(X_train_pca, y_train)
print("KNN best params:", knn.best_params_)


best_model = knn.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Ridge

ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

ridge.fit(X_train_pca, y_train)
print("Ridge best params:", ridge.best_params_)

best_model = ridge.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Lasso

lasso_params = {
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1]
}

lasso = GridSearchCV(
    estimator=Lasso(max_iter=10000),
    param_grid=lasso_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

lasso.fit(X_train_pca, y_train)
print("Lasso best params:", lasso.best_params_)


best_model = lasso.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


'Id=2:BME680:breathVocEquivalent'

In [ ]:
# --- Config ---
n_components = 5         # number of PCA components you chose
test_size = 0.2           # fraction for test set
random_state = 42         # predefined random state for reproducibility

# --- Get arrays (ensure they're numpy arrays) ---
X_arr = np.asarray(X['Id=2:BME680:breathVocEquivalent'])
y_arr = np.asarray(Y['Id=2:BME680:breathVocEquivalent'])


# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=test_size, random_state=random_state, shuffle=True
)

# --- Scale: fit on train, transform train & test ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler.transform(X_test)        # transform only

# --- PCA: fit on scaled train, transform both ---
pca = PCA(n_components=n_components, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# --- SVR: train on PCA data ---
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)  # tune C, epsilon, kernel as needed
svr.fit(X_train_pca, y_train)

# --- Predict & evaluate ---
y_pred = svr.predict(X_test_pca)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE on test set: {mse:.4f}")
print(f"R^2 on test set: {r2:.4f}")

# --- Plot: predicted vs actual ---
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle='--')
plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title("SVR predictions vs actual (after scaling + PCA)")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

svr_params = {
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}

svr = GridSearchCV(
    estimator=SVR(kernel='rbf'),
    param_grid=svr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

svr.fit(X_train_pca, y_train)
print("SVR best params:", svr.best_params_)

from sklearn.metrics import mean_squared_error, r2_score

best_model = svr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________



from sklearn.ensemble import RandomForestRegressor

rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10]
}

rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

rf.fit(X_train_pca, y_train)
print("Random Forest best params:", rf.best_params_)


best_model = rf.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.ensemble import GradientBoostingRegressor

gbr_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

gbr = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gbr_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

gbr.fit(X_train_pca, y_train)
print("GradientBoosting best params:", gbr.best_params_)

best_model = gbr.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.neighbors import KNeighborsRegressor

knn_params = {
    "n_neighbors": [2, 3, 4, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "p": [1, 2]   # Manhattan / Euclidean
}

knn = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=knn_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

knn.fit(X_train_pca, y_train)
print("KNN best params:", knn.best_params_)


best_model = knn.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))


#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Ridge

ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=ridge_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

ridge.fit(X_train_pca, y_train)
print("Ridge best params:", ridge.best_params_)

best_model = ridge.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))

#____________________________________________________
#____________________________________________________
#____________________________________________________

from sklearn.linear_model import Lasso

lasso_params = {
    "alpha": [0.0001, 0.001, 0.01, 0.1, 1]
}

lasso = GridSearchCV(
    estimator=Lasso(max_iter=10000),
    param_grid=lasso_params,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=2
)

lasso.fit(X_train_pca, y_train)
print("Lasso best params:", lasso.best_params_)


best_model = lasso.best_estimator_   # replace with svr, gbr, knn, ridge, lasso…

y_pred = best_model.predict(X_test_pca)

print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R² :", r2_score(y_test, y_pred))
